Terminal code to unzip allele.zip files and rename them with folder name. 
- It will put them in a folder called input. 
- You have to move this folder to another one, for example exon 20, to run the analysis.
- You would input this into the mac terminal. You may need to ask ChatGPT for PC equivalent 

#!/bin/bash

# The directory containing the subfolders
parent_dir="/Users/annahoracek/Desktop/CRISPRessoBatch_on_AH16_X20_green" #location of CRISPResso file
# Loop over each subdirectory
for dir in "$parent_dir"/*; do
    if [ -d "$dir" ]; then
        # Get the name of the subdirectory
        dir_name=$(basename "$dir")

        # Unzip the file and rename it
        for file in "$dir"/Alleles_frequency_table.zip; do
            if [ -f "$file" ]; then
                unzip -o "$file" -d "$dir"
                mv "$dir"/Alleles_frequency_table.txt "$dir"/"$dir_name".txt
            fi
        done

        # Move the renamed file to the done subdirectory
        if [ ! -d "$parent_dir/input" ]; then
            mkdir "$parent_dir/input"
        fi
        mv "$dir"/"$dir_name".txt "$parent_dir/input"
    fi
done

In [9]:
from pandas.core.frame import DataFrame
import pandas as pd
import numpy as np
import itertools
import os
import glob
import re
import os.path
from Bio.Seq import Seq
from Bio import pairwise2
from functools import reduce

In [11]:
base_folder = "/Users/annahoracek/Desktop/Hockemeyer/NGS/2025/MCB153L_reversion_analysis/Exon_5" #location of input files

input_folder = base_folder + "/input" #Folder with text files
output_folder = base_folder + "/output"

#Sample names. Important - Run code sepperately for each group. Names don't work if used together remove or add # to use or silence, respectively.
numbers = ['113_t7', '113_112_t7','113_t21', '113_112_t14','113_mock', '113_112_mock','113_nira', '113_112_nira']
# numbers = ['112_t7', '112_113_t7','112_t21', '112_113_t14','112_mock', '112_113_mock','112_nira', '112_113_nira']

os.makedirs(output_folder, exist_ok=True)

In [12]:
#Function to translate alleles
def append_translation_to_df(df, start_codon, stop_codon):
    # Ensure codons are in uppercase
    start_codon = start_codon.upper()
    stop_codon = stop_codon.upper()

    # Initialize list to store the translation results
    translation_list = []

    for idx, row in df.iterrows():
        # Ensure sequence is in uppercase and remove dashes from the sequence
        seq_no_dash = row['aligned_sequence'].upper().replace("-", "")

        # Find the starting and ending indexes
        start_index = seq_no_dash.find(start_codon)
        end_index = seq_no_dash.find(stop_codon) + len(stop_codon)

        # Cut sequence at this point and begin translating
        seq_to_translate = seq_no_dash[start_index:end_index]

        # Create a Seq object and translate it
        coding_dna = Seq(seq_to_translate)
        protein = coding_dna.translate(table="Standard", to_stop=True)

        # Store the protein sequence in the list
        translation_list.append(str(protein))

    # Append the 'Translation' column to the original DataFrame
    df['Translation'] = translation_list

In [13]:

import glob
import pandas as pd
from functools import reduce
#combine txt files into one csv
def reverse_complement(seq):
    """
    Return the reverse complement of a DNA sequence.
    """
    comp = {
        "A": "T", "T": "A", "C": "G", "G": "C",
        "a": "t", "t": "a", "c": "g", "g": "c",
        "N": "N", "n": "n"
    }
    return "".join(comp.get(base, base) for base in reversed(seq))


# Get all text files from the input folder
file_pattern = os.path.join(input_folder, "*.txt")
file_list = glob.glob(file_pattern)

dfs = []

for file_path in file_list:
    try:
        df = pd.read_csv(file_path, sep="\t")
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        continue

    if "Aligned_Sequence" not in df.columns:
        print(f"Skipping {file_path} because 'Aligned_Sequence' column is missing.")
        continue

    file_name = os.path.splitext(os.path.basename(file_path))[0]

    rename_dict = {}
    if "#Reads" in df.columns:
        rename_dict["#Reads"] = f"{file_name}_#Reads"
    if "%Reads" in df.columns:
        rename_dict["%Reads"] = f"{file_name}_%Reads"

    df = df.rename(columns=rename_dict)

    df["Reverse_Complement"] = df["Aligned_Sequence"].apply(reverse_complement)

    cols_to_keep = [
        "Aligned_Sequence",
        "Reference_Sequence",
        "Reverse_Complement",
        "n_inserted",
        "n_deleted",
        "n_mutated"
    ] + list(rename_dict.values())

    cols_to_keep = [col for col in cols_to_keep if col in df.columns]

    df = df[cols_to_keep]

    dfs.append(df)


if not dfs:
    print("No valid files found. Exiting.")
else:
    merged_df = reduce(
        lambda left, right: pd.merge(
            left,
            right,
            on=[
                "Aligned_Sequence",
                "Reference_Sequence",
                "Reverse_Complement",
                "n_inserted",
                "n_deleted",
                "n_mutated"
            ],
            how="outer"
        ),
        dfs
    )

    merged_df = merged_df.fillna(0)

    reads_cols = [col for col in merged_df.columns if col.endswith("_#Reads")]

    merged_df["common_count"] = merged_df[reads_cols].gt(0).sum(axis=1)
    merged_df["total_reads"] = merged_df[reads_cols].sum(axis=1)

    merged_df = merged_df.sort_values(
        by=["common_count", "total_reads"],
        ascending=False
    )

    merged_df = merged_df.drop(
        columns=["common_count", "total_reads", "Reverse_Complement"],
        errors="ignore"
    )

    merged_df.columns = merged_df.columns.str.lower()

    columns_to_remove = [col for col in merged_df.columns if "%" in col]
    merged_df = merged_df.drop(columns=columns_to_remove)

    os.makedirs(output_folder, exist_ok=True)

    output_csv = os.path.join(output_folder, "merged_ngs_outputs.csv")
    merged_df.to_csv(output_csv, index=False)

    print(f"Merged CSV file saved as: {output_csv}")

    merged_df

Merged CSV file saved as: /Users/annahoracek/Desktop/Hockemeyer/NGS/2025/MCB153L_reversion_analysis/Exon_5/output/merged_ngs_outputs.csv


In [14]:
# Merge all columns and add reads based on passage number and replicate
test = merged_df.copy(deep=True)
def combine_df(df):
    suffixes = ['a', 'b', 'c']

    # Iterate over combinations of weeks, numbers, and suffixes
    for number, suffix in itertools.product(numbers, suffixes):
        # Adjust the pattern to match column names
        pattern = f'{number}_{suffix}'  # Define pattern for current combination

        # Find all column names matching the current pattern
        matching_cols = [col for col in df.columns if re.search(pattern, col)]
        # print(matching_cols)

        # Sum columns matching each pattern and suffix
        if matching_cols:
            df[pattern + '#sum'] = df[matching_cols].sum(axis=1)
            

            # Create percentage column for current suffix
            new_col_name = f'{number}_{suffix}%reads'
            df[new_col_name] = (df[pattern + '#sum'] / df[pattern + '#sum'].sum()) * 100 if df[pattern + '#sum'].sum() != 0 else 0

    # Remove intermediary columns
    columns_to_remove = [col for col in df.columns if '#' in col]
    df.drop(columns=columns_to_remove, inplace=True)

    return df

test =combine_df(test)

#translate 
append_translation_to_df(test, "TTTAGT", "TCAGGT")


test

/Users/annahoracek/opt/anaconda3/lib/python3.9/site-packages/Bio/Seq.py:2804: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


,aligned_sequence,reference_sequence,n_inserted,n_deleted,n_mutated,113_t7_a%reads,113_t7_b%reads,113_t7_c%reads,113_112_t7_a%reads,113_112_t7_b%reads,...,113_112_mock_a%reads,113_112_mock_b%reads,113_112_mock_c%reads,113_nira_a%reads,113_nira_b%reads,113_nira_c%reads,113_112_nira_a%reads,113_112_nira_b%reads,113_112_nira_c%reads,Translation
0,ACCTAAGGGATTTGCTTTGTTTTATTTTAGTCCTGTTGTTCTACAA...,ACCTAAGGGATTTGCTTTGTTTTATTTTAGTCCTGTTGTTCTACAA...,0,0,0,62.056214,61.691774,61.268982,71.199731,73.022463,...,80.952668,80.147765,80.221709,80.109812,81.882429,78.010029,80.981420,81.422205,80.988996,FSPVVLQCTHVTPQRDKSG
2,ACCTAAGG-ATTTGCTTTGTTTTATTTTAGTCCTGTTGTTCTACAA...,ACCTAAGGGATTTGCTTTGTTTTATTTTAGTCCTGTTGTTCTACAA...,0,0,0,1.483055,1.465274,1.279646,1.571077,1.572629,...,1.822163,1.667505,1.879415,1.457416,1.841739,1.643414,1.783198,1.666954,1.718988,FSPVVLQCTHVTPQRDKSG
6,ACCTAAGGGATTTGCTTTGTTTTATTTTAGTCCTGTTGTTCTACAA...,ACCTAAGGGATTTGCTTTGTTTTATTTTAGTCCTGTTGTTCTACAA...,0,0,0,0.535001,0.468744,0.468019,0.495687,0.542488,...,0.556493,0.569559,0.533167,0.592075,0.573937,0.739785,0.612011,0.539236,0.624386,FSPVVLQCTHVTPQRDKSG
8,ACCTAAGGGATTTGCTTTGTTTTATTTTAGTCCTGTTGTTCTACAA...,ACCTAAGGGATTTGCTTTGTTTTATTTTAGTCCTGTTGTTCTACAA...,0,0,0,0.448457,0.486635,0.479867,0.529293,0.565990,...,0.584619,0.601583,0.662016,0.581954,0.627476,0.615660,0.592198,0.561397,0.612823,FSPVVLQCTHVTPQRDKSG
9,ACCTAAGGGATTTGCTTTGTTTTATTTTAGTCCTGTTGTTCTACAA...,ACCTAAGGGATTTGCTTTGTTTTATTTTAGTCCTGTTGTTCTACAA...,0,0,0,0.438622,0.384657,0.371255,0.484485,0.426940,...,0.492205,0.462052,0.526503,0.414959,0.531106,0.446850,0.457908,0.440746,0.464435,FSPVVLQCTHVTPQRDKSG
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24848,TCCTAAGGGATTTGCTTTGTTTTATTTTACTCCTGTTGTTCTACAA...,ACCTAAGGGATTTGCTTTGTTTTATTTTAGTCCTGTTGTTCTACAA...,0,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.002201,0.000000,0.000000,
24849,TCCTAAGGGATTTGCTTTGTTTTATTTTAGTCCTGTTGTTCTACAA...,ACCTAAGGGATTTGCTTTGTTTTATTTTAGTCCTGTTGTTCTACAA...,0,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.002201,0.000000,0.000000,FSPVVLQCTHVTPQRDKSG
24850,TCCTAAGGGATTTGCTTTGTTTTATTTTAGTCCTGTTGTTCTACAA...,ACCTAAGGGATTTGCTTTGTTTTATTTTAGTCCTGTTGTTCTACAA...,0,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.002201,0.000000,0.000000,FSPVVLQCTHVTPQRDKSG
24851,TCCTAAGGGATTTGCTTTGTTTTATTTTAGTCCTGTTGTTCTTCAA...,ACCTAAGGGATTTGCTTTGTTTTATTTTAGTCCTGTTGTTCTACAA...,0,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.002201,0.000000,0.000000,FSPVVLQCTHVTPQRDKSG


In [15]:
#Collapse sequences by transtlation
def sum_and_normalize_reads(df):
    # 1) pick out your “%reads” columns
    read_cols = [col for col in df.columns if col.endswith('%reads')]
    
    # 2) group & sum raw read percentages
    summed = (
        df
        .groupby('Translation', as_index=False)[read_cols]
        .sum()
    )
    
    return summed

average_changed_rev = sum_and_normalize_reads(test)
average_changed_rev

,Translation,113_t7_a%reads,113_t7_b%reads,113_t7_c%reads,113_112_t7_a%reads,113_112_t7_b%reads,113_112_t7_c%reads,113_t21_a%reads,113_t21_b%reads,113_t21_c%reads,...,113_mock_c%reads,113_112_mock_a%reads,113_112_mock_b%reads,113_112_mock_c%reads,113_nira_a%reads,113_nira_b%reads,113_nira_c%reads,113_112_nira_a%reads,113_112_nira_b%reads,113_112_nira_c%reads
0,,11.162251,10.695245,12.217856,6.049065,4.567086,5.155460,3.134916,2.373691,2.918864,...,2.769830,1.966811,1.818473,2.081575,2.001417,1.895278,8.291545,1.972526,1.854086,1.873157
1,FS,0.000000,0.000000,0.000000,0.000000,0.054836,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,FSAVVLQCTHVTPQRDKSG,0.000000,0.001789,0.000000,0.002800,0.003917,0.001895,0.002250,0.003885,0.000000,...,0.002345,0.000000,0.000000,0.000000,0.000000,0.002142,0.000000,0.002201,0.000000,0.000000
3,FSCNTTKR,0.000000,0.000000,0.000000,0.000000,0.013709,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,FSCSTMYTCNTTKR,0.000000,0.000000,0.000000,0.000000,0.000000,0.001895,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1688,FSVTPQRDKSG,0.000000,0.000000,0.000000,0.000000,0.000000,0.009473,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.004443,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1689,FSVVILQCTHVTPQRDKSG,0.000000,0.000000,0.000000,0.000000,0.001958,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1690,FSYM,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.002222,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1691,FSYNIHM,0.000000,0.000000,0.000000,0.005601,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [16]:
#Identify reversions 

example_df_rev = average_changed_rev.copy()

mask = (
    ~example_df_rev['Translation'].str.endswith('S', na=False)
) & (
    ~example_df_rev['Translation'].str.endswith('*', na=False)
)


# append a "*" to those entries
example_df_rev.loc[mask, 'Translation'] += '*'

# your WT
wt = "PVVLQCTHVTPQRDKS"

def align_sequences(wt_seq: str, mut_seq: str):
    """Return a pair of aligned strings (with '-' for gaps) by
    stripping off common prefix/suffix and padding in the middle."""
    # 1) longest common prefix
    start = 0
    while start < len(wt_seq) and start < len(mut_seq) and wt_seq[start] == mut_seq[start]:
        start += 1
    # 2) longest common suffix (without overlapping prefix)
    end = 0
    while end < len(wt_seq) - start and end < len(mut_seq) - start \
          and wt_seq[-(end+1)] == mut_seq[-(end+1)]:
        end += 1
    # 3) extract middles
    wt_mid  = wt_seq[start:len(wt_seq)-end]
    mut_mid = mut_seq[start:len(mut_seq)-end]
    # 4) pad shorter middle with dashes
    if len(wt_mid) > len(mut_mid):
        mut_mid += "-" * (len(wt_mid) - len(mut_mid))
    elif len(mut_mid) > len(wt_mid):
        wt_mid  += "-" * (len(mut_mid) - len(wt_mid))
    # 5) rebuild full aligned sequences
    aligned_wt  = wt_seq[:start] + wt_mid + wt_seq[len(wt_seq)-end:]
    aligned_mut = mut_seq[:start] + mut_mid + mut_seq[len(mut_seq)-end:]
    return aligned_wt, aligned_mut

#trim translations
def trim_translation_endings(
    df: pd.DataFrame,
    translation_col: str = "Translation",
    five_prime_trim: str = "FS",
    three_prime_trim: str = "GM"
) -> pd.DataFrame:
    """
    Trim the given motifs from the start and/or end of each sequence in `translation_col`.
    - If starts with `five_prime_trim`, drop it.
    - If ends with `three_prime_trim` ("GM"), drop both.
    - Else if ends with "G" or "M", drop that single letter.
    """
    def _trim_seq(seq):
        if not isinstance(seq, str):
            return seq
        # 5′ trim
        if seq.startswith(five_prime_trim):
            seq = seq[len(five_prime_trim):]
        # 3′ trim: first look for "GM"
        if seq.endswith(three_prime_trim):
            seq = seq[:-len(three_prime_trim)]
        # if not GM but ends in G or M, trim one
        elif seq.endswith("G") or seq.endswith("M"):
            seq = seq[:-1]
        return seq

    df[translation_col] = df[translation_col].apply(_trim_seq)
    return df

example_df_rev = trim_translation_endings(
    average_changed_rev,
    translation_col="Translation",
    five_prime_trim="FS",
    three_prime_trim="GM"
)

def classify_translation(trans: str, wt: str) -> str:
    # 0) never forget to guard empties
    if not trans:
        return "Other"
    # 2) frameshift (doesn't end in Y)
    if  trans.endswith("*"):
        return "Frameshift"

    # 3) exact WT
    if trans == wt:
        return "Wild-type"

    # 1) single‐AA substitution (missense)
    if len(trans) == len(wt):
        mismatches = sum(1 for a, b in zip(trans, wt) if a != b)
        if mismatches < 2:
            return "Missense"
    if len(trans) == len(wt):
        mismatches = sum(1 for a, b in zip(trans, wt) if a != b)
        if mismatches >= 2:
            return "seq_error"    

    # 4) align and count events
    awt, amat = align_sequences(wt, trans)
    deletions  = sum(1 for a, b in zip(awt, amat) if a != "-" and b == "-")
    insertions = sum(1 for a, b in zip(awt, amat) if a == "-" and b != "-")
    subs       = sum(1 for a, b in zip(awt, amat) if a != "-" and b != "-" and a != b)

    # 5) pure in‐frame deletion
    if deletions > 0 and insertions == 0 and subs == 0 and trans.endswith("S"):
        return "In_frame_deletion"

    # 6) complex IF: deletion + <3 subs, no insertions
    if deletions > 0 and insertions == 0 and subs < 3 and trans.endswith("S"):
        return "Complex_in_frame"

    # 7) everything else is a reversion
    if trans.endswith("S") and len(trans) <= len(wt) and "THV" not in trans:
        return "Reversion"
    
    return "Complex"


# apply to your DataFrame
example_df_rev['type'] = ""
example_df_rev['type'] = (
    example_df_rev['Translation']
      .fillna("")  # guard NaNs
      .astype(str)
      .apply(lambda t: classify_translation(t, wt))
)

#save file
output_csv = os.path.join(output_folder, "translated_alleles.csv")
example_df_rev.to_csv(output_csv, index=False)
print(f"Merged CSV file saved as: {output_csv}")

#simplify allele catagorization
# 2. mapping of the three rows into “in-frame category”
map_dict = {
    'Missense':               'In-frame',
    'In_frame_deletion':      'In-frame',
    'Complex_in_frame':       'In-frame',
    'Complex':  "Other",
    "seq_error": "Other"
    
}

# 3. replace the type names
example_df_rev['type'] = example_df_rev['type'].replace(map_dict)

# 4. collapse (here we sum the %reads across those three; you could use .mean() if you prefer)
example_df_rev = (
    example_df_rev
    .groupby('type', as_index=False)
    .sum()
)

output_csv = os.path.join(output_folder, "allele_type_summary.csv")
example_df_rev.to_csv(output_csv, index=False)
print(f"Merged CSV file saved as: {output_csv}")


example_df_rev

Merged CSV file saved as: /Users/annahoracek/Desktop/Hockemeyer/NGS/2025/MCB153L_reversion_analysis/Exon_5/output/translated_alleles.csv
Merged CSV file saved as: /Users/annahoracek/Desktop/Hockemeyer/NGS/2025/MCB153L_reversion_analysis/Exon_5/output/allele_type_summary.csv


/var/folders/md/qj_hgg097yq2z5t14lb2pm280000gn/T/ipykernel_93979/3515740439.py:153: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  example_df_rev


,type,113_t7_a%reads,113_t7_b%reads,113_t7_c%reads,113_112_t7_a%reads,113_112_t7_b%reads,113_112_t7_c%reads,113_t21_a%reads,113_t21_b%reads,113_t21_c%reads,...,113_mock_c%reads,113_112_mock_a%reads,113_112_mock_b%reads,113_112_mock_c%reads,113_nira_a%reads,113_nira_b%reads,113_nira_c%reads,113_112_nira_a%reads,113_112_nira_b%reads,113_112_nira_c%reads
0,In-frame,3.314254,3.236483,3.082604,5.214518,4.435871,4.670418,3.416226,4.840621,3.817851,...,3.834608,4.851736,5.533190,5.022882,4.883356,3.835528,1.494464,4.935717,4.661069,4.551849
1,Other,25.176531,25.662862,26.513162,12.686233,11.494095,12.817598,6.006526,4.817311,5.361681,...,3.370233,2.501205,2.516126,2.892433,2.978088,2.398544,8.485179,2.326964,2.400709,2.416604
2,Reversion,0.000000,0.007156,0.001975,0.050409,0.050919,0.020842,0.000000,0.013597,0.000000,...,0.000000,0.044198,0.086921,0.059981,0.000000,0.002142,0.000000,0.004403,0.032009,0.061668
3,Wild-type,71.509215,71.093498,70.402259,82.048841,84.019114,82.491142,90.577248,90.328471,90.820468,...,92.795159,92.602861,91.863763,92.024703,92.138556,93.763786,90.020356,92.732917,92.906212,92.969879
